# Reproduction - Deep Transformer Models for Time Series Forecasting: The Influenza Prevalence Case

## Load Dataset

In [2]:
from dataset import Dataset

dataset = Dataset(path="./data/state/ILINet.csv")
train_loader, vali_loader, test_loader = dataset.get_train_val_test_loader()

Train samples: 16052
Validation samples: 1784
Test samples: 4460


D:\Courses\ECE228\epidemic-transformer-forecasting\dataset.py:49: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  torch.tensor(x_train, dtype=torch.float32).unsqueeze(-1),


## Deep Transformer Model

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import matplotlib.pyplot as plt


class PositionalEncoding(nn.Module):
    """
    Adds sinusoidal positional encoding to each token's features so that
    the Transformer is position-aware.

    Reference: Vaswani et al. 2017, Section 3.5
      PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
      PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))
    """
    def __init__(self, d_model: int, max_len: int = 1000):
        super(PositionalEncoding, self).__init__()

        ###########################################################################
        # Pre-compute the sinusoidal encoding matrix and register
        # it as a buffer of shape [1, max_len, d_model].
        ###########################################################################

        pe = torch.zeros(max_len, d_model)              # [max_len, d_model]
        position = torch.arange(0, max_len).unsqueeze(1) # [max_len, 1]

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, D]
        returns: [B, T, D] with positional encoding added.
        """
        pe = None
        ###########################################################################
        # Add the positional encoding to x.
        ###########################################################################

        T = x.size(1)
        x = x + self.pe[:, :T, :]
        return x


class DeepTransformer(nn.Module):
    def __init__(self, input_dim = 1: int, output_dim = 1: int, d_model: int = 64, num_heads: int = 4,
                 ff_dim: int = 6, encoder_layers: int = 4, decoder_layers: int = 4,
                 dropout: float = 0.1):
        super(DeepTransformer, self).__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.position_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=False
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=encoder_layers
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=False
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=decoder_layers
        )

        self.output_proj = nn.Linear(d_model, output_dim)

    def generate_square_subsequent_mask(self, T, device):
        """
        upper triangle matrix:
          0  -inf -inf -inf
          0    0  -inf -inf
          0    0    0  -inf
          0    0    0    0
        """
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float("-inf"))
        return mask

    def foward(self, src, tgt):
        """
        src: [B, src_len, input_dim]
             example: x1...x10

        tgt: [B, tgt_len, input_dim]
             example: x10...x13

        return: [B, tgt_len, output_dim]
             example: predict x11...x14
        """
        src = self.input_proj(src)
        src = self.position_encoder(src)

        memory = self.encoder(src)  # self attention

        tgt = self.input_proj(tgt)
        tgt = self.position_encoder(tgt)

        tgt_len = tgt.size(1)
        tgt_mask = self.generate_square_subsequent_mask(tgt_len, tgt.device)

        out = self.decoder(tgt=tgt, memory=memory, tgt_mask=tgt_mask)

        return out

In [4]:
class PipeLine:
    def __init__(self, model, train_loader, val_loader, warmup_steps=5000, d_model=64):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.d_model = d_model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.warmup_steps = warmup_steps
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            betas=(0.9, 0.98),
            eps=1e-9
        )
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=self.lr_lambda
        )
        self.loss_func = nn.MSELoss()

    def lr_lambda(self, step):
        step = max(step, 1)
        lr = (self.d_model ** (-0.5)) * min(step ** -0.5, step * (self.warmup_steps ** -1.5))
        return lr

    def train_model(self, epochs):
        train_losses = []
        val_losses = []
    
        best_val_loss = float("inf")
        best_model = None

        for epoch in range(1, epochs+1):
            train_loss = self.train_one_epoch()
            train_losses.append(train_loss)
    
            val_loss = self.evaluate(self.val_loader)
            val_losses.append(val_loss)
    
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model = copy.deepcopy(self.model)
    
            if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
                print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        return best_model, train_losses, val_losses
    
    def train_one_epoch(self):
        """
        Run one full pass over the training dataloader and return the average loss.
        """
        self.model.train()
        total_loss, total_count = 0.0, 0
        for xb, yb in self.train_loader:
            xb, yb = xb.to(self.device), yb.to(self.device)
            self.optimizer.zero_grad()
    
            pred = None
            pred = self.model(xb)
            loss = self.loss_func(pred, yb)
            loss.backward()
    
            self.optimizer.step()
            self.scheduler.step()
            total_loss  += loss.item() * xb.size(0)
            total_count += xb.size(0)
        return total_loss / total_count

    @torch.no_grad()
    def evaluate(self, dataloader):
        """
        Evaluate the model on valid dataloader without updating weights.
        Returns the average loss.
        """
        self.model.eval()
        total_loss, total_count = 0.0, 0
    
        for xb, yb in dataloader:
            xb, yb = xb.to(self.device), yb.to(self.device)
            pred = self.model(xb)
            loss = self.loss_func(pred, yb)
    
            total_loss  += loss.item() * xb.size(0)
            total_count += xb.size(0)
        return total_loss / total_count

# Improvement - Deep Transformer Models for Time Series Forecasting: The Influenza Prevalence Case